# Лаба 1: Логистическая регрессия
Запуск по примеру одногруппника.

*(ключевой момент: используем готовый файл data/titanic_preprocessed.csv, поэтому менять working directory не нужно)*


In [1]:
import sys, os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
os.environ.setdefault('MPLCONFIGDIR', 'outputs/mpl_cache')
os.makedirs('outputs/mpl_cache', exist_ok=True)

from src.data.preprocessing import preprocess_titanic_data
from src.models.logistic_regression import LogisticRegression
from src.utils.visualization import plot_training_curves, plot_confusion_matrix, plot_feature_importance

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)
np.random.seed(42)


In [2]:

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

X_train, X_test, y_train, y_test, feature_names = preprocess_titanic_data('../datasets/titanic_preprocessed.csv', device=device)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Features: {X_train.shape[1]}")
print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Classes in y_train: {torch.unique(y_train)}")
print(f"Class distribution in y_train: {torch.bincount(y_train.long())}")
print(f"Class distribution in y_test: {torch.bincount(y_test.long())}")

print("X_train stats:")
print(f"Min: {X_train.min().item():.4f}")
print(f"Max: {X_train.max().item():.4f}")
print(f"Mean: {X_train.mean().item():.4f}")
print(f"Std: {X_train.std().item():.4f}")


PyTorch version: 2.8.0
CUDA available: False
Device: cpu
X_train shape: torch.Size([712, 7])
X_test shape: torch.Size([177, 7])
y_train shape: torch.Size([712])
y_test shape: torch.Size([177])
Features: 7
Train samples: 712
Test samples: 177
Classes in y_train: tensor([0., 1.])
Class distribution in y_train: tensor([445, 267])
Class distribution in y_test: tensor([104,  73])
X_train stats:
Min: -2.2222
Max: 9.9716
Mean: -0.0000
Std: 1.0001


In [3]:

learning_rate = 0.005
max_epochs = 3000

model = LogisticRegression(
    learning_rate=learning_rate,
    max_epochs=max_epochs,
    device=device,
    reg_type=None,  # можно попробовать 'l2' или 'elasticnet'
    l1_lambda=0.0,
    l2_lambda=0.0,
    alpha=0.5,
)

print(f"Learning rate: {learning_rate}")
print(f"Max epochs: {max_epochs}")

model.fit(X_train, y_train, X_test, y_test)
print(f"Epochs: {len(model.history['train_loss'])}")
if model.history['val_metrics']:
    print(f"Final val accuracy: {model.history['val_metrics'][-1]['accuracy']:.4f}")

plot_training_curves(model.history, title='Logistic Regression Training Curves', model_type='classification', save_path='../outputs/logreg/logreg_training_curves.png')


Learning rate: 0.005
Max epochs: 3000
Epochs: 3000
Final val accuracy: 0.8136


In [4]:

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

test_metrics = model.score(X_test, y_test)
print('Test results:')
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall: {test_metrics['recall']:.4f}")
print(f"  F1-Score: {test_metrics['f1_score']:.4f}")

baseline_pred = torch.full_like(y_test, torch.mode(y_train)[0])
baseline_accuracy = torch.mean((y_test == baseline_pred).float()).item()
print('Baseline comparison:')
print(f"  Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"  Accuracy improvement: {((test_metrics['accuracy'] - baseline_accuracy) / baseline_accuracy * 100):.1f}%")


Test results:
  Accuracy: 0.8136
  Precision: 0.8125
  Recall: 0.7123
  F1-Score: 0.7591
Baseline comparison:
  Baseline Accuracy: 0.5876
  Accuracy improvement: 38.5%


In [5]:

plot_confusion_matrix(y_test.cpu(), y_pred.cpu(), title='Confusion Matrix', save_path='../outputs/logreg/confusion_matrix.png')

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(y_pred_proba.cpu().numpy(), bins=20, alpha=0.7, edgecolor='black')
plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold = 0.5')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.title('Predicted Probabilities Distribution')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
survived_probs = y_pred_proba[y_test == 1].cpu().numpy()
died_probs = y_pred_proba[y_test == 0].cpu().numpy()

plt.hist(died_probs, bins=20, alpha=0.7, label='Died (y=0)', color='red')
plt.hist(survived_probs, bins=20, alpha=0.7, label='Survived (y=1)', color='green')
plt.axvline(x=0.5, color='black', linestyle='--', label='Threshold = 0.5')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.title('Probability Distribution by True Class')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('../outputs/logreg/probability_analysis.png', dpi=200)
plt.close()


In [6]:

weights = model.weights.detach().cpu().numpy()
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Weight': weights.reshape(-1),
    'Abs_Weight': np.abs(weights).reshape(-1)
}).sort_values('Abs_Weight', ascending=False)
print(feature_importance)
plot_feature_importance(weights, feature_names, title='Feature Importance (Absolute Weights)', save_path='../outputs/logreg/feature_importance.png')


            Feature    Weight  Abs_Weight
5       Sex_encoded -1.119596    1.119596
0            Pclass -0.656792    0.656792
1               Age -0.257803    0.257803
2             SibSp -0.236550    0.236550
4              Fare  0.195868    0.195868
3             Parch -0.135502    0.135502
6  Embarked_encoded  0.005960    0.005960
